In [28]:
import zipfile
import os

zip_path = "/content/cleaned_data.zip"
extract_path = "/content/cleaned_data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

os.listdir(extract_path)

['sellers.csv',
 'order_items.csv',
 'order_reviews.csv',
 'customers.csv',
 'geolocation.csv',
 'products.csv',
 'order_payments.csv',
 'product_category.csv',
 'orders.csv']

In [29]:
!pip install -q "psycopg[binary]" --quiet

In [30]:
from google.colab import userdata
database_url = userdata.get('Analyst_NeonDB_Postgresql_server')

In [31]:
import psycopg

conn = psycopg.connect(database_url)

print("Connected to Neon successfully.")

Connected to Neon successfully.


In [32]:
with conn.cursor() as cur:
  cur.execute("SELECT version();")
  print(cur.fetchone()[0])

PostgreSQL 18.6 (c5250a2) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


Postgresql connection established

In [33]:
import os

data_dir = "/content/cleaned_data"

csv_files = [
    f for f in os.listdir(data_dir)
    if f.endswith(".csv")
]

csv_files

['sellers.csv',
 'order_items.csv',
 'order_reviews.csv',
 'customers.csv',
 'geolocation.csv',
 'products.csv',
 'order_payments.csv',
 'product_category.csv',
 'orders.csv']

In [34]:
import pandas as pd

csv_files = [f.removesuffix(".csv") for f in csv_files]

for file_name in csv_files:
  df = pd.read_csv(f"{data_dir}/{file_name}.csv")
  globals()[file_name] = df
  print(f"{file_name} : shape = {df.shape}")

sellers : shape = (3095, 4)
order_items : shape = (112650, 7)
order_reviews : shape = (99224, 7)
customers : shape = (99441, 5)
geolocation : shape = (1000163, 5)
products : shape = (32950, 9)
order_payments : shape = (103886, 5)
product_category : shape = (71, 2)
orders : shape = (99441, 8)


In [ ]:
from sqlalchemy import create_engine

engine = create_engine(database_url)

for file_name in csv_files:
    df_to_upload = globals()[file_name]
    df_to_upload.to_sql(
        file_name, 
        engine,
        if_exists="replace", 
        index=False 
    )
    print(f"'{file_name}' table uploaded successfully.")

print("All tables uploaded successfully!")

'sellers' table uploaded successfully.
'order_items' table uploaded successfully.
'order_reviews' table uploaded successfully.
'customers' table uploaded successfully.
'geolocation' table uploaded successfully.
'products' table uploaded successfully.
'order_payments' table uploaded successfully.
'product_category' table uploaded successfully.
'orders' table uploaded successfully.
All tables uploaded successfully!


In [36]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tables = pd.read_sql(query, engine)
tables

,table_name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category
7,products
8,sellers


In [37]:
for table_name in tables['table_name']:

    query = f"""
    SELECT COUNT(*) AS row_count
    FROM {table_name};
    """

    count = pd.read_sql(query, engine).iloc[0]["row_count"]

    print(f"{table_name:20s} {count:,}")

customers            99,441
geolocation          1,000,163
order_items          112,650
order_payments       103,886
order_reviews        99,224
orders               99,441
product_category     71
products             32,950
sellers              3,095


All tables loaded in NeonDB

## Analytics



```
customers
    │
    │ customer_id
    ↓
orders
    │
    ├──────────────→ order_reviews
    │
    ├──────────────→ order_payments
    │
    ↓ order_id
order_items
    │
    ├────────→ products
    │
    └────────→ sellers
```



In [38]:
# Order Status

query = f"""
select order_status, COUNT(*) as order_count
from orders
group by order_status
order by order_count DESC;"""

order_status = pd.read_sql(query, engine)
order_status

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [39]:
# Unique customers
query = "select count(distinct customer_unique_id) as unique_customers from customers;"
unique_customers = pd.read_sql(query, engine)
unique_customers

,unique_customers
0,96096


In [40]:
# order date range
query = """
SELECT
    MIN(order_purchase_timestamp) AS first_order,
    MAX(order_purchase_timestamp) AS last_order
FROM orders;
"""

date_range = pd.read_sql(query, engine)
date_range

,first_order,last_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


In [41]:
query = """
SELECT
    DATE_TRUNC('month', order_purchase_timestamp::timestamp) AS month,
    COUNT(*) AS orders
FROM orders
GROUP BY 1
ORDER BY 1;
"""

monthly_orders = pd.read_sql(query, engine)
monthly_orders

,month,orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


In [42]:
import plotly.express as px

fig = px.line(
    monthly_orders,
    x="month",
    y="orders",
    title="Monthly Order Volume",
    labels={
        "month": "Month",
        "orders": "Number of Orders"
    }
)

fig.update_xaxes(tickangle=45)
fig.show()

In [43]:
# product sales value
query = """
SELECT
    SUM(price) AS total_product_sales
FROM order_items;
"""

sales = pd.read_sql(query, engine)

display(sales)
sales.iloc[0]['total_product_sales']


,total_product_sales
0,1.359164e+07


np.float64(13591643.70001419)

In [44]:
# freight charges
query = """
SELECT
    SUM(freight_value) AS total_freight_value
FROM order_items;
"""

freight = pd.read_sql(query, engine)
freight

,total_freight_value
0,2251909.54


In [45]:
# sales + freight
query = """
SELECT
    SUM(price) AS product_sales,
    SUM(freight_value) AS freight_value,
    SUM(price + freight_value) AS merchandise_plus_freight
FROM order_items;
"""

sales_summary = pd.read_sql(query, engine)
display(sales_summary)
print('merchandise_plus_freight: ',sales_summary.iloc[0]['merchandise_plus_freight'])

,product_sales,freight_value,merchandise_plus_freight
0,1.359164e+07,2251909.54,1.584355e+07


merchandise_plus_freight:  15843553.239998734


In [46]:
# Average Order Value
query = """
SELECT
    SUM(price) / COUNT(DISTINCT order_id) AS average_order_value
FROM order_items;
"""

aov = pd.read_sql(query, engine)
aov

,average_order_value
0,137.754076


In [47]:
# Monthly sales (time + sales)

query = """select date_trunc('month', o.order_purchase_timestamp::timestamp) as month,
  sum(oi.price) as product_sales, count(distinct o.order_id) as orders,
  sum(oi.price)/count(distinct o.order_id) as average_order_value
  from orders as o join order_items as oi on o.order_id=oi.order_id
  group by 1
  order by 1;"""

monthly_sales = pd.read_sql(query, engine)
monthly_sales

,month,product_sales,orders,average_order_value
0,2016-09-01,267.36,3,89.120000
1,2016-10-01,49507.66,308,160.739156
2,2016-12-01,10.90,1,10.900000
3,2017-01-01,120312.87,789,152.487795
4,2017-02-01,247303.02,1733,142.702262
5,2017-03-01,374344.30,2641,141.743393
6,2017-04-01,359927.23,2391,150.534182
7,2017-05-01,506071.14,3660,138.270803
8,2017-06-01,433038.60,3217,134.609450
9,2017-07-01,498031.48,3969,125.480343


In [48]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=("Monthly Product Sales", "Monthly Average Order Value"))

# Add line graph for Monthly Product Sales
fig.add_trace(
    go.Scatter(
        x=monthly_sales["month"],
        y=monthly_sales["product_sales"],
        mode="lines+markers",
        name="Product Sales",
        marker=dict(color='blue')
    ),
    row=1, col=1
)

# Add bar graph for Monthly Average Order Value
fig.add_trace(
    go.Bar(
        x=monthly_sales["month"],
        y=monthly_sales["average_order_value"],
        name="Average Order Value",
        marker=dict(color='green')
    ),
    row=2, col=1
)

# Update layout
fig.update_layout(
    title_text="Monthly Sales Trends",
    height=700,
    showlegend=False
)

fig.update_xaxes(tickangle=45, title_text="Month", row=2, col=1)
fig.update_yaxes(title_text="Total Product Sales", row=1, col=1)
fig.update_yaxes(title_text="Average Order Value", row=2, col=1)

fig.show()

In [49]:
query = """ select * from products as p join product_category as pc on p.product_category_name=pc.product_category_name;
"""
p = pd.read_sql(query, engine)
p.drop(columns=['product_category_name'], inplace=True)
col = p.pop('product_category_name_english')
p.insert(1, 'product_category_name_english', col)
p

,product_id,product_category_name_english,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,art,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,sports_leisure,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,baby,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,housewares,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32323,a0b7d5a992ccda646f2d34e418fff5a0,furniture_decor,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32324,bf4538d88321d0fd4412a93c974510e6,construction_tools_lights,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32325,9a7c6041fa9592d9d9ef6cfe62a71f8c,bed_bath_table,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32326,83808703fc0706a22e264b9d75f04a2e,computers_accessories,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [50]:
p.to_sql('products', engine, if_exists='replace', index=False)
products = pd.read_sql('select * from products;', engine)
print("Products table updated!")

Products table updated!
